<a href="https://colab.research.google.com/github/faresaouadi/project_data_analysis/blob/main/project_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA Analysis Project: Binary Classification
## Predicting Income Level Using the Adult Dataset

---

### **Project Identification Card:**
- **Student:** Fares Aouadi
- **Academic Level:** First-Year Master's in Artificial Intelligence (Master 1 AI)
- **Institution:** University of El Oued (Algeria)
- **Group:** [2]

---

### **Project Abstract:**
This applied project aims to build and deploy a specialized machine learning model for **Binary Classification** to predict whether an individual's income exceeds $50K annually, using the benchmark `adult.csv` dataset. The workflow encompasses Exploratory Data Analysis (EDA), data cleaning and handling, categorical variable encoding, model training, and comprehensive performance evaluation.

## **Phase 1: Data Acquisition & EDA**

### **Objective:**
Load the dataset from the repository and perform an exploratory data analysis to inspect its structure, data types, missing values, and target class balance.

### **Key Steps:**
1. **Load Data:** Fetch the dataset file (CSV) from the repository and read it into the workspace using `pd.read_csv()`.
2. **Inspect Structure & Dimensions:** Verify the number of rows and columns, and determine the data types of each column (numerical or text) using `df.shape` and `df.info()`.
3. **Detect Missing Values & Anomalies:** Scan columns for missing values (`NaN`) or illogical zeros using `df.isnull().sum()`.
4. **Analyze Target Balance:** Calculate the proportion of Class 0 versus Class 1 using `value_counts()` to check if the data is balanced or suffers from Class Imbalance.

In [33]:
import pandas as pd


column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

df = pd.read_csv("/content/sample_data/adult.csv", header=None, names=column_names)
print(df.head())
print("="*50)
print("First 5 rows of the dataset:")
print("="*50)
print(df.head())
print("="*50)
print("Dataset information:")
print("="*50)


print(df.info())
print("="*50)
print("Summary statistics:")
print("="*50)
print(df.describe())
print("="*50)
print("Missing values:")
print("="*50)
print(df.isnull().sum())

     age  workclass  fnlwgt  education  education-num  marital-status  \
0  48842         14     NaN        NaN            NaN             NaN   
1    age  workclass  fnlwgt  education  education-num  marital-status   
2     39          8   77516         10             13               5   
3     50          7   83311         10             13               3   
4     38          5  215646         12              9               1   

   occupation  relationship  race  sex  capital-gain  capital-loss  \
0         NaN           NaN   NaN  NaN           NaN           NaN   
1  occupation  relationship  race  sex  capital-gain  capital-loss   
2           2             2     5    2          2174             0   
3           5             1     5    2             0             0   
4           7             2     5    2             0             0   

   hours-per-week  native-country   income  
0             NaN             NaN      NaN  
1  hours-per-week  native-country  incomes  
2    

In [34]:
print("="*50)
print("Class balance:")
print("="*50)

print(df['income'].iloc[2:].value_counts())

Class balance:
income
1    37155
2    11687
Name: count, dtype: int64


## **Phase 2: Data Preprocessing & Cleaning**

### **Objective:**
Clean the dataset by handling missing values/anomalies, encoding categorical text features into numerical formats, scaling numerical features, and separating features from the target variable.

### **Key Steps:**
1. **Handle Missing Values (Imputation):** Drop rows with severe missing data or replace missing/unknown values (such as `?` in the Adult dataset) using the mean/median for numerical columns or the mode (most frequent value) for categorical columns.
2. **Categorical Encoding:** Convert text features into numerical values using `LabelEncoder` for binary/ordinal variables, and `OneHotEncoder` for nominal categorical variables.
3. **Feature Scaling:** Standardize the range of numerical features using `StandardScaler` or `MinMaxScaler` to prevent large-magnitude features from dominating the model.
4. **Separate Features and Target:** Isolate the feature matrix ($X$) from the target variable ($y$), and drop any unnecessary ID or serial columns.

In [35]:
import numpy as np


df.replace(' ?', np.nan, inplace=True)
df.replace('?', np.nan, inplace=True)

df.dropna(inplace=True)

print("data clean", df.shape)
print("="*50)
print("Missing values:")
print("="*50)
print(df.isnull().sum())

data clean (48843, 15)
Missing values:
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64


In [36]:

from sklearn.preprocessing import LabelEncoder, StandardScaler

df = df[df['income'] != 'incomes']

numeric_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

le = LabelEncoder()
y = le.fit_transform(df['income'])

X = df.drop(columns=['income'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.select_dtypes(include=['int64', 'float64']))


print(df['income'].value_counts())

income
1    37155
2    11687
Name: count, dtype: int64


## Phase 3: Train/Test Split and Baseline Model

In this phase, we partition the preprocessed dataset into training and testing subsets to evaluate model generalization on unseen data, then establish an initial performance benchmark using a baseline classifier.

### Key Steps:
1. **Stratified Data Splitting:** Divide the feature matrix ($X$) and target vector ($y$) into **80% training** and **20% testing** sets using `train_test_split` with `stratify=y` to preserve the original class distribution across both subsets.
2. **Baseline Model Training:** Train an initial **Logistic Regression** model (`max_iter=1000`) on the training data to serve as a reference benchmark for future comparisons.
3. **Initial Evaluation:** Measure performance using key evaluation metrics including **Accuracy**, **Precision**, **Recall**, and **F1-Score**, alongside generating the **Confusion Matrix**.

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)


(39073, 6) (9769, 6)


LogisticRegression(max_iter=1000, random_state=42)

In [38]:

y_pred = baseline_model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='binary')
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')
conf_matrix = confusion_matrix(y_test, y_pred)


print("=== Baseline Logistic Regression Metrics ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)

=== Baseline Logistic Regression Metrics ===
Accuracy:  0.8142
Precision: 0.6930
Recall:    0.4016
F1-Score:  0.5085

Confusion Matrix:
[[7015  416]
 [1399  939]]


## Phase 4: Model Development and Systematic Comparison

In this phase, we go beyond our linear baseline by training advanced ensemble learning algorithms capable of capturing complex, non-linear relationships within the dataset, followed by a systematic performance comparison.

### Key Steps:
1. **Advanced Ensemble Modeling:** Train and evaluate stronger tree-based ensemble models such as **Random Forest** (`RandomForestClassifier`), **XGBoost**, and **LightGBM** to improve predictive power over the baseline.
2. **Systematic Comparison:** Compare all models against the Logistic Regression baseline using **Accuracy**, **Precision**, **Recall**, and **F1-Score** to select the most stable model with the optimal balance between precision and recall.

In [39]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

rf_model = RandomForestClassifier(random_state=42)
xgb_model = XGBClassifier(random_state=42)
lgbm_model = LGBMClassifier(random_state=42)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
lgbm_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)
y_pred_lgbm = lgbm_model.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
precision_rf = precision_score(y_test, y_pred_rf, average='binary')
precision_xgb = precision_score(y_test, y_pred_xgb, average='binary')
precision_lgbm = precision_score(y_test, y_pred_lgbm, average='binary')
recall_rf = recall_score(y_test, y_pred_rf, average='binary')
recall_xgb = recall_score(y_test, y_pred_xgb, average='binary')
recall_lgbm = recall_score(y_test, y_pred_lgbm, average='binary')
f1_rf = f1_score(y_test, y_pred_rf, average='binary')
f1_xgb = f1_score(y_test, y_pred_xgb, average='binary')
f1_lgbm = f1_score(y_test, y_pred_lgbm, average='binary')


results_data = {
    'Model': ['Logistic Regression (Baseline)', 'Random Forest', 'XGBoost', 'LightGBM'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_lgbm)
    ],
    'Precision': [
        precision_score(y_test, y_pred, average='binary'),
        precision_score(y_test, y_pred_rf, average='binary'),
        precision_score(y_test, y_pred_xgb, average='binary'),
        precision_score(y_test, y_pred_lgbm, average='binary')
    ],
    'Recall': [
        recall_score(y_test, y_pred, average='binary'),
        recall_score(y_test, y_pred_rf, average='binary'),
        recall_score(y_test, y_pred_xgb, average='binary'),
        recall_score(y_test, y_pred_lgbm, average='binary')
    ],
    'F1-Score': [
        f1_score(y_test, y_pred, average='binary'),
        f1_score(y_test, y_pred_rf, average='binary'),
        f1_score(y_test, y_pred_xgb, average='binary'),
        f1_score(y_test, y_pred_lgbm, average='binary')
    ]
}


comparison_df = pd.DataFrame(results_data)


comparison_df = comparison_df.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("="*70)
print("(Systematic Comparison Table)")
print("="*70)
display(comparison_df)

[LightGBM] [Info] Number of positive: 9349, number of negative: 29724
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002134 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 616
[LightGBM] [Info] Number of data points in the train set: 39073, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.239270 -> initscore=-1.156685
[LightGBM] [Info] Start training from score -1.156685
(Systematic Comparison Table)


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1-Score
0,XGBoost,0.845941,0.787440,0.488024,0.602588
1,LightGBM,0.844611,0.785914,0.482036,0.597561
2,Random Forest,0.807759,0.616044,0.522241,0.565278
3,Logistic Regression (Baseline),0.814208,0.692989,0.401625,0.508530


## Phase 5: Model Optimization and Hyperparameter Tuning

In this final optimization phase, we fine-tune our best-performing model (`XGBoost`) to maximize predictive performance and ensure robust generalization on unseen data.

### Key Steps:
1. **Hyperparameter Tuning:** Use `RandomizedSearchCV` to explore optimal combinations of key hyperparameters (such as `n_estimators`, `max_depth`, and `learning_rate`) to boost the F1-Score.
2. **Stratified 5-Fold Cross-Validation (`cv=5`):** Implement robust cross-validation to evaluate model stability across multiple data folds, preventing overfitting and confirming reliable generalization.

In [40]:
from sklearn.model_selection import RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
xgb_base = XGBClassifier(random_state=42)
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

print("Best Hyperparameters:", random_search.best_params_)
best_xgb = random_search.best_estimator_
y_pred_best_xgb = best_xgb.predict(X_test)
print("Best XGBoost Metrics:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_best_xgb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best_xgb, average='binary'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_best_xgb, average='binary'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_best_xgb, average='binary'):.4f}")

Best Hyperparameters: {'subsample': 1.0, 'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Best XGBoost Metrics:
Accuracy:  0.8478
Precision: 0.7978
Recall:    0.4876
F1-Score:  0.6053


## Phase 6: Model Persistence, Inference, and Documentation

In this final phase, we export our optimized classification model for production readiness, validate its real-time prediction capability on unseen sample data, and prepare professional documentation for repository hosting.

### Key Steps:
1. **Model Persistence:** Serialize and save the tuned **XGBoost** model alongside the fitted `StandardScaler` and `LabelEncoder` as `.pkl` files using the `joblib` library.
2. **Inference Testing:** Build a standalone prediction function (`predict_income`) that loads the saved artifacts and performs end-to-end inference on a new individual's data sample.
3. **Project Documentation:** Structure the repository with the cleaned `.ipynb` notebook, model artifacts, and a comprehensive `README.md` summarizing the problem, preprocessing pipeline, and model comparison results.

In [41]:
import joblib
best_model = random_search.best_estimator_
joblib.dump(best_model, 'best_xgb_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')
#

['label_encoder.pkl']

In [42]:
import joblib
import pandas as pd

def predict_new_individual(sample_data_dict):
    """
A function to test a saved model on a new sample of data
    """

    model = joblib.load('best_xgb_model.pkl')
    scaler = joblib.load('scaler.pkl')
    le = joblib.load('label_encoder.pkl')


    input_df = pd.DataFrame([sample_data_dict])

    scaled_data = scaler.transform(input_df)

    pred_encoded = model.predict(scaled_data)[0]
    pred_proba = model.predict_proba(scaled_data)[0]


    original_label = le.inverse_transform([pred_encoded])[0]
    confidence = pred_proba[pred_encoded] * 100

    print("="*40)
    print(" Prediction results for the new sample:")
    print(f"- Expected category: {original_label}")
    print(f"- Confidence level: {confidence:.2f}%")
    print("="*40)

sample_data = {
    'age': 38,
    'fnlwgt': 215646,
    'education-num': 9,
    'capital-gain': 0,
    'capital-loss': 0,
    'hours-per-week': 40
}


predict_new_individual(sample_data)

 Prediction results for the new sample:
- Expected category: 1
- Confidence level: 83.74%


## Project Summary, Key Findings and Conclusion :

This final section summarizes the entire machine learning pipeline, highlights the key experimental results, and outlines the final conclusions drawn from our binary classification study on the `adult.csv` dataset.

### Summary of Experiments and Results:
1. **Baseline Model (`LogisticRegression`):** Served as our initial reference benchmark, achieving an Accuracy of `81.42%` and an F1-Score of `0.5085`.
2. **Advanced Ensemble Models:** Training tree-based classifiers significantly boosted predictive power.
   - **Random Forest:** Improved the F1-Score to `0.5653`.
   - **XGBoost & LightGBM:** Delivered superior performance, with **XGBoost** standing out as the most balanced classifier (Accuracy: `84.59%`, F1-Score: `0.6026`).
3. **Hyperparameter Optimization (`Tuned XGBoost`):** Applying `RandomizedSearchCV` with 5-fold cross-validation (`cv=5`) successfully fine-tuned the model parameters (`n_estimators=400`, `max_depth=5`, `learning_rate=0.1`), raising the final Accuracy to `84.78%` and F1-Score to `0.6053`.

### Conclusion:
This comprehensive, step-by-step methodology—spanning from exploratory data analysis and preprocessing to systematic model comparison and hyperparameter tuning—proves the clear superiority of tree-based ensemble methods over linear models for tabular demographic datasets. The successful serialization of the model artifacts (`.pkl`) and validation via the custom inference pipeline mark a complete and production-ready machine learning lifecycle for this academic project.